In [4]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
import sys
from notebookutils import mssparkutils

StatementMeta(, b24ad0d8-a13d-4b31-9766-13cac562cf50, 6, Finished, Available, Finished)

## Enable Spark Logger

In [5]:
logger = spark._jvm.org.apache.log4j.LogManager.getLogger("FabricLogger")

StatementMeta(, b24ad0d8-a13d-4b31-9766-13cac562cf50, 7, Finished, Available, Finished)

## Invoking File Parameters From Pipeline

In [ ]:
NB_fullFilepath = ""
schema_path = ""
schema_file_name = ""
bronze_lakehouse_id = ""
bronze_workspace_id = ""

print(bronze_lakehouse_id)


## Reading schema File for Load

In [2]:
try:
    # df_ref_schema = spark.read.csv(schema_path + schema_file_name, header=True)

    #schema_full_path = ("abfss://" + bronze_workspace_id + "@onelake.dfs.fabric.microsoft.com/" + bronze_lakehouse_id + "/" + schema_path + "/"+ schema_file_name)
    schema_full_path = (
    "abfss://" +
    bronze_workspace_id +
    "@onelake.dfs.fabric.microsoft.com/" +
    bronze_lakehouse_id +
    "/" +
    schema_path +
    "/" +
    schema_file_name
)


    # df_ref_schema = spark.read.csv(schema_full_path, header=True)
    print(schema_full_path)
    df_ref_schema = spark.read.csv(schema_full_path, header=True)
    logger.info("Schema file extract successfully")
except Exception as e:
    logger.error(f"error loading schema file: {str(e)}")
    raise e

StatementMeta(, 1b6fa9b1-2d0b-49c4-a22f-4a8c8a328a52, 4, Finished, Available, Finished)

abfss://@onelake.dfs.fabric.microsoft.com///


NameError: name 'logger' is not defined

## Schema file count check

In [5]:
count = df_ref_schema.count()

if count == 0:
    raise Exception("Schema file empty")
display(count)

StatementMeta(, b4c25dcb-4f33-4e4c-b8ae-9c3274ecb9bf, 7, Finished, Available, Finished)

11

## Extract metadata from Source Bronze layer File

In [6]:
try:
    src_path = ("abfss://"+ bronze_workspace_id+ "@onelake.dfs.fabric.microsoft.com/" + bronze_lakehouse_id + "/" + NB_fullFilepath)
    print(src_path)
    print(NB_fullFilepath)
    df = spark.read.csv(src_path, header=True, inferSchema=True)
   # df = spark.read.csv(NB_fullFilepath, header=True, inferSchema=True)
    # df = df.toDF(*[c.lower() for c in df.columns])
    file_schema = df.schema
    print(file_schema)
    logger.info("Source file extract successfully")
except Exception as e:
    logger.error(f"error source loading file: {str(e)}")
    raise e
    print(f"error source loading file: {e}")

StatementMeta(, b4c25dcb-4f33-4e4c-b8ae-9c3274ecb9bf, 8, Finished, Available, Finished)

StructType([StructField('OrderID', StringType(), True), StructField('OrderDate', DateType(), True), StructField('CustomerID', StringType(), True), StructField('CustomerName', StringType(), True), StructField('ProductCategory', StringType(), True), StructField('ProductName', StringType(), True), StructField('Quantity', IntegerType(), True), StructField('UnitPrice', IntegerType(), True), StructField('Region', StringType(), True), StructField('SalesChannel', StringType(), True), StructField('SalesAmount', IntegerType(), True)])


## Construncting Structure Schema From Schema File

In [7]:
from pyspark.sql.types import *

def sql_to_spark_type(sql_type):
    t = sql_type.lower().split('(')[0]

    mapping = {
        "string": StringType(),
        "varchar": StringType(),
        "char": StringType(),
        "int": IntegerType(),
        "integer": IntegerType(),
        "bigint": LongType(),
        "double": DoubleType(),
        "float": DoubleType(),
        "decimal": DoubleType(),
        "boolean": BooleanType(),
        "date": DateType(),
        "timestamp": TimestampType(),
        "datetime": TimestampType()
    }

    return mapping.get(t, StringType())   # default safety


StatementMeta(, b4c25dcb-4f33-4e4c-b8ae-9c3274ecb9bf, 9, Finished, Available, Finished)

In [8]:
ref_schema = StructType([
    StructField(
        row.column_name,
        sql_to_spark_type(row.data_type),
        True
    )
    for row in df_ref_schema.collect()
])
print(ref_schema)

StatementMeta(, b4c25dcb-4f33-4e4c-b8ae-9c3274ecb9bf, 10, Finished, Available, Finished)

StructType([StructField('OrderID', StringType(), True), StructField('OrderDate', DateType(), True), StructField('CustomerID', StringType(), True), StructField('CustomerName', StringType(), True), StructField('ProductCategory', StringType(), True), StructField('ProductName', StringType(), True), StructField('Quantity', IntegerType(), True), StructField('UnitPrice', IntegerType(), True), StructField('Region', StringType(), True), StructField('SalesChannel', StringType(), True), StructField('SalesAmount', IntegerType(), True)])


## Comparing Reference Schema with File Schema

In [9]:
if ref_schema == file_schema:
    print("Schema is valid")
else:
    raise Exception("schema mismatch detected.")

StatementMeta(, b4c25dcb-4f33-4e4c-b8ae-9c3274ecb9bf, 11, Finished, Available, Finished)

Schema is valid
